# Traffic Congestion Classification: Comparing ML Algorithms

**Title:** Comparing Classical Machine Learning and Deep Learning Models for Traffic Congestion Classification on Smart City Sensor Data.

**Research question:** How do classical ML algorithms (SVM, Random Forest, XGBoost, ...) compare with deep learning models (neural networks) for classifying traffic congestion level (Low / Medium / High) from multi-sensor smart-city data, in terms of accuracy, training time, and model size?

---
### How to use this notebook
This notebook is a **template**. Every code cell below is a set of instructions and `# TODO:` comments. **You write the code.** Work top to bottom, one TODO at a time. Ask your instructor if you get stuck.

**Heads up:** the sensor CSVs are small (tens of rows). That is fine for learning the whole ML pipeline, but your numbers will be noisy and the neural networks will tend to overfit. Noticing *why* small data hurts deep learning is part of the project, so keep notes for your report.

# Step 1 - Setup + Download
*(Weeks 1-2)*

In this step you set up your tools, download the sensor data, and look at the four sensor tables so you understand what you are working with.

## Week 1 - Environment & first look

**Goals:** mount Google Drive, install libraries, download the smart-city sensor CSVs, and load your first table.

In [ ]:
# Mount Google Drive so your files are saved between sessions.
# TODO: import the `drive` module from google.colab and call drive.mount('/content/drive')
# TODO: choose a project folder inside your Drive and save its path in a variable, e.g. PROJECT_DIR


In [ ]:
# Install / import the libraries you will need throughout the project.
# TODO: if needed, pip install xgboost (most others are pre-installed on Colab)
# TODO: import numpy, pandas, matplotlib.pyplot
# TODO: import tensorflow and print its version


In [ ]:
# Check what hardware you have (optional - the neural nets here are tiny, so CPU is fine).
# TODO: use tensorflow to list physical GPU devices and print them


In [ ]:
# Download the smart-city sensor dataset.
# TODO: git clone https://github.com/rauffauzanrambe/smart-city-sensor-data-2026 into your project folder
# TODO: save the path to the data/raw folder in a variable, e.g. DATA_DIR


In [ ]:
# "Hello data": load the traffic sensor table and look at it.
# TODO: read DATA_DIR/traffic_sensors.csv into a pandas DataFrame
# TODO: show the first few rows (.head()) and print its shape (rows, columns)


## Week 2 - Understand the data

**Goals:** load all four sensor tables, understand the columns and zones, and look at how the readings vary.

In [ ]:
# Load all four sensor tables into separate DataFrames.
# The files are: traffic_sensors.csv, air_quality_sensors.csv, weather_station.csv, energy_grid.csv
# Tip: bad readings are stored as the text "NaN", so pass na_values=['NaN'] to read_csv.
# TODO: read each CSV (with na_values=['NaN']) and print its shape + column names


In [ ]:
# Explore the traffic table.
# TODO: count rows per zone (value_counts on 'location')
# TODO: count rows per status (value_counts on 'status') - how many are ERROR?
# TODO: describe() the traffic_count column (min/max/mean)


In [ ]:
# Visualize the readings.
# TODO: bar chart of how many traffic readings each zone has
# TODO: histogram of traffic_count (drop the missing values first)


# Step 2 - Data Preprocessing
*(Weeks 3-4)*

Here you clean the data, **time-align the four sensors into one table**, build a congestion label, split it into train/val/test, and prepare inputs for both classical ML and the neural networks.

## Week 3 - Clean, align, label & split

**Goals:** drop bad readings, parse timestamps, join each traffic reading to the nearest weather and air-quality reading in the same zone, turn traffic counts into a Low/Medium/High label, and split into train/validation/test.

In [ ]:
# Clean the tables and parse timestamps.
# A reading is "bad" if its status is ERROR or its main value is missing (NaN).
# TODO: convert the 'timestamp' column to datetime (pd.to_datetime) in each table
# TODO: drop rows with a missing main value: traffic_count in traffic, pm25/co2 in air,
#       temperature in weather
# TODO: sort each table by timestamp (needed for the time-align step next)


In [ ]:
# Time-align the sensors: give each traffic reading the closest weather + air-quality
# reading from the SAME zone. pandas merge_asof matches each row to the nearest earlier
# timestamp; using by='location' keeps it within the same zone.
# TODO: merge_asof(traffic, weather[...], on='timestamp', by='location', direction='nearest')
#       keep weather columns: temperature, humidity, wind_speed, precipitation
# TODO: merge_asof the result with air[...] the same way
#       keep air columns: pm25, co2
# TODO: fill any leftover gaps in the merged sensor columns with the column mean (.fillna)
# TODO: print the merged shape and .head()


In [ ]:
# Build the congestion label from traffic_count.
# Split the counts into three equal-sized groups: Low, Medium, High.
# TODO: use pd.qcut(df['traffic_count'], 3, labels=['Low','Medium','High']) -> df['congestion']
# TODO: print the count of each congestion class


In [ ]:
# Split into train / validation / test (e.g. 60% / 20% / 20%), stratified by congestion.
# "Stratified" keeps the class balance the same in each split.
# The dataset is small, so expect only a handful of rows per split - that is okay.
# TODO: train_test_split TWICE (peel off test, then split the rest into train/val),
#       passing stratify=the congestion label, random_state=42
# TODO: add a column 'split' to df marking each row as train/val/test


## Week 4 - Feature table + scaled arrays

You will prepare data **two ways**:
1. **Classical features** - a table of numbers per reading (sensor values + which zone) for sklearn/XGBoost.
2. **Scaled arrays** - the same features, standardized (mean 0, std 1), which neural networks train on better.

In [ ]:
# Choose the feature columns and turn the zone into numbers.
# Features: temperature, humidity, wind_speed, precipitation, pm25, co2, and the zone.
# (We do NOT use traffic_count as a feature - it is what the label is built from!)
# TODO: one-hot encode the 'location' column (pd.get_dummies) and pick the feature columns
# TODO: build a list FEATURES of the column names you will use


In [ ]:
# Build X (features) and y (label) for each split, then save them.
# TODO: encode the congestion label to integers with LabelEncoder
# TODO: for each split, take X = df[FEATURES][rows in split], y = encoded label
# TODO: np.savez to PROJECT_DIR/features.npz so you don't recompute every time
# TODO: print the shapes of X_train, X_val, X_test


In [ ]:
# Scale the features for the neural networks.
# Neural nets train better when every feature has mean 0 and std 1.
# TODO: fit a StandardScaler on X_train ONLY, then transform train/val/test
# TODO: save the scaled arrays (e.g. features_scaled.npz) for Week 7


In [ ]:
# Quick EDA on the prepared data.
# TODO: confirm the class balance in each split (how many Low/Medium/High per split)
# TODO (optional): correlation heatmap of the numeric features (df[FEATURES].corr())


# Step 3 - Training & Evaluation
*(Weeks 5-8)*

Now you train many models and compare them. Keep ONE shared results table and add a row for every model so the final comparison is easy.

In [ ]:
# Create one shared results table you will append to all through Step 3.
# TODO: make an empty list called `results`
# TODO: write a small helper that, given (name, y_true, y_pred, train_time),
#       computes accuracy and macro-F1 (sklearn.metrics) and appends a dict to results


## Week 5 - Classical models, round 1

Train four simple classifiers on the feature table from Week 4: Logistic Regression, K-Nearest Neighbors, Decision Tree, and Naive Bayes.

In [ ]:
# Load the features you saved in Week 4.
# TODO: np.load PROJECT_DIR/features.npz and pull out X_train, y_train, X_val, y_val, ...


In [ ]:
# Logistic Regression.
# TODO: import LogisticRegression, fit on (X_train, y_train), time the fit
# TODO: predict on X_val and record the result in your results table


In [ ]:
# K-Nearest Neighbors.
# TODO: import KNeighborsClassifier, fit, predict on val, record result
# TODO (variation): try a few values of n_neighbors (e.g. 3, 5) and see what changes


In [ ]:
# Decision Tree.
# TODO: import DecisionTreeClassifier, fit, predict, record result
# TODO (variation): try different max_depth values


In [ ]:
# Naive Bayes.
# TODO: import GaussianNB, fit, predict, record result


In [ ]:
# Show a confusion matrix for your best model so far.
# TODO: use sklearn.metrics.ConfusionMatrixDisplay on val predictions


## Week 6 - Classical models, round 2 + tuning

Train stronger models (SVM, Random Forest, XGBoost) and try **hyperparameter variations** to improve them.

In [ ]:
# Support Vector Machine (SVM).
# TODO: import SVC, train with a linear kernel and again with an 'rbf' kernel
# TODO (variation): try a few values of C; record each result


In [ ]:
# Random Forest.
# TODO: import RandomForestClassifier, fit, predict, record result
# TODO (variation): try different n_estimators and max_depth


In [ ]:
# XGBoost.
# TODO: import XGBClassifier, fit, predict, record result
# TODO (variation): try different n_estimators / max_depth / learning_rate


In [ ]:
# Compare your classical models so far.
# TODO: turn `results` into a DataFrame and sort by accuracy
# TODO: print it and note which classical model is best


## Week 7 - Neural network from scratch

Build your own small neural network (a Multi-Layer Perceptron, or MLP) and train it on the scaled features. Watch what happens with so little data.

In [ ]:
# Load the SCALED features and build a small MLP.
# An MLP stacks Dense layers; the last layer has 3 units + softmax (one score per class).
# TODO: np.load PROJECT_DIR/features_scaled.npz for the scaled X arrays (reuse y from before)
# TODO: build a tf.keras Sequential: Input -> Dense(16, relu) -> Dense(8, relu) -> Dense(3, softmax)
# TODO: compile (optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# TODO: print model.summary()


In [ ]:
# Train the MLP.
# TODO: model.fit(Xs_train, y_train, validation_data=(Xs_val, y_val), epochs=..., verbose=0) and time it
# TODO: keep the History object to plot curves next


In [ ]:
# Plot training curves.
# TODO: plot training vs validation accuracy across epochs
# TODO: plot training vs validation loss across epochs
# Look for overfitting: training accuracy climbing while validation accuracy stalls or drops.


In [ ]:
# Save the model and record its result.
# TODO: model.save to PROJECT_DIR
# TODO: get validation predictions (argmax of model.predict) and append to your results table


## Week 8 - Deeper/wider network + final comparison

Try a bigger neural network with dropout, then compare EVERYTHING. With a small dataset, watch whether the bigger network actually helps or just overfits.

In [ ]:
# Deeper / wider MLP with dropout (dropout randomly turns off neurons to fight overfitting).
# TODO: build a bigger network, e.g. Dense(64) -> Dropout -> Dense(32) -> Dropout -> Dense(3, softmax)
# TODO: train it, time it, record its validation result


In [ ]:
# One more variation of your choice.
# TODO: try a different size, more/fewer epochs, or a different optimizer; record the result


In [ ]:
# FINAL COMPARISON of every model you trained.
# TODO: build the results DataFrame and sort by accuracy / F1
# TODO: bar chart of accuracy per model
# TODO: scatter plot of accuracy vs training time


In [ ]:
# Error analysis on your single best model.
# TODO: pick the best model, predict on the validation set
# TODO: confusion matrix; note which congestion levels get confused with each other


In [ ]:
# Final score: evaluate the best model on the TEST set (do this only once, at the end).
# The test set was never used for training or tuning, so it gives an honest final number.
# TODO: predict on X_test and report accuracy + macro-F1


# Step 4 - Write Report
*(Weeks 9-10)*

Turn your results into a clear story, and build a small demo.

## Week 9 & 10 - Draft & demo

**Goals:** draft the report and build a "predict congestion from a new reading" demo.

### Report draft (write in these markdown cells)

- **Introduction:** Why does predicting traffic congestion in a smart city matter?
- **Dataset:** What sensors are in the smart-city data? How did you align them and build the label?
- **Methods:** What features and what models did you try?
- **Results:** Your comparison table and charts. Which model won?
- **Discussion:** Why might a simple model beat a neural network here? How did the small dataset affect your results? What would you do with more data?

*TODO: write your draft here as you go.*

In [ ]:
# Prediction demo: type in a sensor reading and let your best model guess the congestion level.
# TODO: build one row of features in the SAME column order as FEATURES (and scale it if your
#       best model is the neural network)
# TODO: run best.predict and print the predicted congestion class
